In [4]:
import kfp

from kfp import dsl

from kfp.components import func_to_container_op, InputPath, OutputPath


@func_to_container_op
def read_data_file(input_path: InputPath(),
                   output_text_path: OutputPath(str)):
    print("step in add_double")
    f = open(input_path)
    line1 = f.readline()
    print(line1)
    line2 = f.readline()
    print(line2)
    with open(output_text_path, 'w') as writer:
        writer.write(str(line1 + "连接字符" + line2))


@kfp.dsl.pipeline(
    name='Titantic training pipeline',
    description='My machine learning pipeline'
)
def titantic_pipeline():
    vop = dsl.VolumeOp(
        name="volume_creation",
        resource_name="titantic_pvc",
        storage_class="local-storage",
        modes=["ReadWriteOnce"],
        size="1Gi",
        volume_name="my-pv"
    )

    def produce_data_op(volume):
        return dsl.ContainerOp(
            name="Titantic-Data",
            image="ubuntu:18.04",
            file_outputs={
                'data': '/home/dlf/kubeflow-test/pipeline/loadFileTest/test/data/test.txt'
            },
            pvolumes={"/data": volume}
        )

    produce_data_task = produce_data_op(vop.volume)
    print(produce_data_task.outputs)
    inputFilePath = produce_data_task.outputs['data']
    read_data_file(inputFilePath)


kfp.compiler.Compiler().compile(titantic_pipeline, "readfile" + '.yaml')

{'data': {{pipelineparam:op=titantic-data;name=data}}}
